In [5]:
MODEL_PATH = "/data/aneesh/gpt_oss_120b"
MAPPING_PATH = "/data/aneesh/aimo/pruning/mappings/expert_mapping_120b_mxfp4_new_large.json"
SAVE_PATH = "./models/threshold_98_5"

THRESHOLD = 98.5

OVERWRITE_OUTPUT = True

import os 
os.environ["CUDA_VISIBLE_DEVICES"] = "0,1"

import json
import math
import shutil
from pathlib import Path

import torch
from safetensors.torch import safe_open, save_file
from transformers import AutoTokenizer

# ── Helpers ──

MODEL_PATH = Path(MODEL_PATH)
MAPPING_PATH = Path(MAPPING_PATH)
SAVE_PATH = Path(SAVE_PATH)

def ensure_output_dir(path: Path, overwrite: bool = False):
    if path.exists():
        existing = list(path.iterdir())
        if existing and not overwrite:
            raise RuntimeError(f"Output dir already exists and is non-empty: {path}")
        if overwrite:
            shutil.rmtree(path)
    path.mkdir(parents=True, exist_ok=True)

def load_json(path: Path):
    with open(path, "r") as f:
        return json.load(f)

def save_json(obj, path: Path):
    with open(path, "w") as f:
        json.dump(obj, f, indent=2)

def list_source_shards(model_path: Path):
    index_path = model_path / "model.safetensors.index.json"
    if index_path.exists():
        index = load_json(index_path)
        shard_names = sorted(set(index["weight_map"].values()))
        return shard_names, index
    shard_names = sorted([p.name for p in model_path.glob("*.safetensors")])
    return shard_names, None

def tensor_nbytes(t):
    return t.numel() * t.element_size()

# ── Core Threshold Pruning Logic ──

def build_selected_experts(stats_json, threshold=99.0):
    num_layers = stats_json["num_layers"]
    
    selected = {}
    old_to_new = {}
    summary_stats = []

    print(f"\n{'Layer':<8} {'Total Experts':<16} {'Needed for {:.1f}%'.format(threshold):<20} {'% of Experts Used':<20} {'Never Used':<12}")
    print("-" * 78)

    for layer_idx in range(num_layers):
        stats = stats_json["layer_stats"][f"layer_{layer_idx}"]
        counts = stats["selection_counts"]
        total_events = stats["selection_events"]

        total_experts = len(counts)
        
        # Pair counts with indices to track which experts we are keeping
        expert_usage = [(idx, count) for idx, count in enumerate(counts)]
        expert_usage.sort(key=lambda x: x[1], reverse=True)

        cumulative = 0.0
        needed = 0
        chosen_indices = []

        for idx, count in expert_usage:
            chosen_indices.append(idx)
            needed += 1
            if total_events > 0:
                cumulative += 100.0 * count / total_events
            if cumulative >= threshold:
                break

        never_used = sum(1 for c in counts if c == 0)
        pct_used = 100.0 * needed / total_experts

        # Sort indices back to original order to retain sequential memory layout
        chosen_indices = sorted(chosen_indices)

        selected[layer_idx] = chosen_indices
        old_to_new[str(layer_idx)] = {str(old_idx): new_idx for new_idx, old_idx in enumerate(chosen_indices)}

        print(f"{layer_idx:<8} {total_experts:<16} {needed:<20} {pct_used:<19.2f}% {never_used:<12}")
        summary_stats.append({
            "layer": layer_idx,
            "total_experts": total_experts,
            "needed": needed,
            "pct_used": pct_used,
            "never_used": never_used,
        })

    # ── Per-layer needed vs total bar ──
    print(f"\n── Per-Layer: Needed vs Total Experts (threshold={threshold:.1f}%) ──\n")
    bar_width = 40
    for s in summary_stats:
        filled = int(round(bar_width * s["needed"] / s["total_experts"]))
        bar = "█" * filled + "░" * (bar_width - filled)
        print(f"  Layer {s['layer']:>3} | {bar} | {s['needed']:>4} / {s['total_experts']} ({s['pct_used']:.1f}%)")

    # ── Aggregate Summary ──
    total_layers        = len(summary_stats)
    total_experts_all   = sum(s["total_experts"] for s in summary_stats)
    total_needed_all    = sum(s["needed"] for s in summary_stats)
    total_never_used    = sum(s["never_used"] for s in summary_stats)
    avg_needed          = total_needed_all / total_layers
    avg_pct             = sum(s["pct_used"] for s in summary_stats) / total_layers
    min_s               = min(summary_stats, key=lambda s: s["needed"])
    max_s               = max(summary_stats, key=lambda s: s["needed"])
    overall_pct         = 100.0 * total_needed_all / total_experts_all

    print(f"\n── Aggregate Summary ──")
    print(f"  Threshold                  : {threshold:.1f}%")
    print(f"  Total layers               : {total_layers}")
    print(f"  Total experts (all layers) : {total_experts_all}")
    print(f"  Total needed   (all layers): {total_needed_all}  ({overall_pct:.2f}% of all experts)")
    print(f"  Total never used           : {total_never_used}")
    print(f"  Avg experts needed / layer : {avg_needed:.1f}  ({avg_pct:.2f}% of layer total)")
    print(f"  Min needed                 : {min_s['needed']} experts  (Layer {min_s['layer']}, {min_s['pct_used']:.1f}%)")
    print(f"  Max needed                 : {max_s['needed']} experts  (Layer {max_s['layer']}, {max_s['pct_used']:.1f}%)")

    total_prunable = total_experts_all - total_needed_all
    print(f"\n── Pruning Potential ──")
    print(f"  Experts that can be pruned : {total_prunable} / {total_experts_all}  ({100.0 * total_prunable / total_experts_all:.2f}% reduction)\n")

    return selected, old_to_new, summary_stats

# ── Slicing & File IO ──

def is_prunable_key(key: str):
    suffixes = [
        ".mlp.router.weight",
        ".mlp.router.bias",
        ".mlp.experts.gate_up_proj_bias",
        ".mlp.experts.down_proj_bias",
        ".mlp.experts.gate_up_proj_blocks",
        ".mlp.experts.gate_up_proj_scales",
        ".mlp.experts.down_proj_blocks",
        ".mlp.experts.down_proj_scales",
    ]
    return any(key.endswith(s) for s in suffixes)

def extract_layer_idx(key: str):
    parts = key.split(".")
    for i in range(len(parts) - 2):
        if parts[i] == "layers":
            return int(parts[i + 1])
    raise ValueError(f"Could not parse layer idx from key: {key}")

def slice_expert_dim0(tensor: torch.Tensor, selected_experts):
    idx = torch.tensor(selected_experts, dtype=torch.long)
    return tensor.index_select(0, idx)

def maybe_adjust_router_bias(key: str, tensor: torch.Tensor, current_layer_experts: int):
    # Adjusts based on the specific number of experts retained in this layer
    if key.endswith(".mlp.router.bias") and current_layer_experts < 4:
        adjustment = math.log(4.0 / float(current_layer_experts))
        return tensor + adjustment
    return tensor

def rewrite_config(model_path: Path, save_path: Path, selected_by_layer: dict):
    config_path = model_path / "config.json"
    config = load_json(config_path)

    # Record the variable number of experts per layer for your optimization logic
    experts_per_layer = [len(selected_by_layer[i]) for i in range(len(selected_by_layer))]
    config["num_local_experts_per_layer"] = experts_per_layer
    
    # Update base fields (using max as a fallback for standard architecture requirements)
    config["num_local_experts"] = max(experts_per_layer)
    min_experts = min(experts_per_layer)
    config["num_experts_per_tok"] = min(config.get("num_experts_per_tok", 4), min_experts)

    save_json(config, save_path / "config.json")
    return config

def copy_non_weight_files(model_path: Path, save_path: Path):
    skip_names = {"config.json", "model.safetensors.index.json"}
    for p in model_path.iterdir():
        if p.name in skip_names or p.suffix == ".safetensors":
            continue
        dst = save_path / p.name
        if p.is_file():
            shutil.copy2(p, dst)
        elif p.is_dir():
            shutil.copytree(p, dst, dirs_exist_ok=True)

# ── Main Execution ──

# 1. Build pruning plan
mapping_json = load_json(MAPPING_PATH)
selected_by_layer, expert_mapping_used, pruning_summary = build_selected_experts(mapping_json, THRESHOLD)

# 2. Prune and write new safetensor shards
ensure_output_dir(SAVE_PATH, overwrite=OVERWRITE_OUTPUT)
shard_names, source_index = list_source_shards(MODEL_PATH)

new_weight_map = {}
total_size = 0
written_tensor_count = 0
pruned_tensor_count = 0
copied_tensor_count = 0

print("Pruning shards...")
for shard_name in shard_names:
    src_shard_path = MODEL_PATH / shard_name
    dst_shard_path = SAVE_PATH / shard_name

    out_tensors = {}

    with safe_open(str(src_shard_path), framework="pt", device="cpu") as f:
        for key in f.keys():
            tensor = f.get_tensor(key)

            if is_prunable_key(key):
                layer_idx = extract_layer_idx(key)
                selected = selected_by_layer[layer_idx]
                
                new_tensor = slice_expert_dim0(tensor, selected)
                new_tensor = maybe_adjust_router_bias(key, new_tensor, len(selected))
                
                pruned_tensor_count += 1
            else:
                new_tensor = tensor
                copied_tensor_count += 1

            out_tensors[key] = new_tensor.contiguous()
            new_weight_map[key] = shard_name
            total_size += tensor_nbytes(new_tensor)
            written_tensor_count += 1

    save_file(out_tensors, str(dst_shard_path))

print("\n── Write Summary ──")
print("written_tensor_count:", written_tensor_count)
print("pruned_tensor_count: ", pruned_tensor_count)
print("copied_tensor_count: ", copied_tensor_count)
print("num_output_shards:   ", len(shard_names))

# 3. Write index json, config, tokenizer, and mapping used
index_payload = {
    "metadata": {"total_size": int(total_size)},
    "weight_map": new_weight_map
}
save_json(index_payload, SAVE_PATH / "model.safetensors.index.json")

new_config = rewrite_config(MODEL_PATH, SAVE_PATH, selected_by_layer)

tokenizer = AutoTokenizer.from_pretrained(str(MODEL_PATH), trust_remote_code=True)
tokenizer.save_pretrained(str(SAVE_PATH))

copy_non_weight_files(MODEL_PATH, SAVE_PATH)
save_json(expert_mapping_used, SAVE_PATH / "expert_mapping.json")

final_summary = {
    "source_model_path": str(MODEL_PATH),
    "mapping_path": str(MAPPING_PATH),
    "save_path": str(SAVE_PATH),
    "threshold": THRESHOLD,
    "num_layers": len(selected_by_layer),
    "num_output_shards": len(shard_names),
    "written_tensor_count": written_tensor_count,
    "pruned_tensor_count": pruned_tensor_count,
    "copied_tensor_count": copied_tensor_count,
    "layer_stats": pruning_summary
}

save_json(final_summary, SAVE_PATH / "pruning_summary.json")

print("\nSaved to:", SAVE_PATH)
print("Config 'num_local_experts' (fallback max):", new_config["num_local_experts"])
print("Config 'num_experts_per_tok':", new_config["num_experts_per_tok"])
print("Index tensors mapped:", len(new_weight_map))


Layer    Total Experts    Needed for 98.5%     % of Experts Used    Never Used  
------------------------------------------------------------------------------
0        128              117                  91.41              % 0           
1        128              113                  88.28              % 0           
2        128              106                  82.81              % 0           
3        128              104                  81.25              % 0           
4        128              106                  82.81              % 0           
5        128              110                  85.94              % 0           
6        128              110                  85.94              % 0           
7        128              105                  82.03              % 0           
8        128              105                  82.03              % 0           
9        128              103                  80.47              % 0           
10       128              97 


── Write Summary ──
written_tensor_count: 687
pruned_tensor_count:  288
copied_tensor_count:  399
num_output_shards:    15

Saved to: models/threshold_98_5
Config 'num_local_experts' (fallback max): 117
Config 'num_experts_per_tok': 4
Index tensors mapped: 687
